# VAAYU — train on Colab GPU

Runs against the shared Google Drive folder `MyDrive/PS2-VAAYU` (all data already there):
**collect CAMS → preprocess → graph → train → evaluate → promote**.

Setup (once):
1. `Runtime → Change runtime type → T4 GPU`.
2. Colab **Secrets** (🔑, left sidebar): add `ADS_KEY` (Copernicus ADS key) and enable notebook access.
3. Set `REPO_URL` in the first cell.

Run the cells top to bottom. Nothing here overwrites raw data; processed files and models are rebuilt.

In [ ]:
# 1. Mount Drive, get the code, install what Colab doesn't ship (torch is preinstalled)
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/kn0wbodyRules/PS2-VAAYU.git"
REPO_DIR = "/content/PS2-VAAYU"

import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q torch-geometric xarray netCDF4 cdsapi duckdb python-dotenv pyarrow fastapi

In [ ]:
# 2. Keys from Colab Secrets -> environment (config.py reads them at import)
from google.colab import userdata
for key in ["ADS_KEY", "OPENAQ_API_KEY", "FIRMS_MAP_KEY"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"secret {key} not set (only ADS_KEY is needed for this notebook)")

import torch, config
config.ensure_dirs()
print("environment:", config.ENVIRONMENT, "| drive root:", config.DRIVE_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")

In [ ]:
# 3. Split. DEADLINE split = 2025-focused (the full 2015-2025 CAMS archive is still downloading).
#    Windows without CAMS are skipped automatically, so any 2024 that has arrived is used too.
DEADLINE_SPLIT = True
if DEADLINE_SPLIT:
    config.TRAIN_RANGES = [("2024-01-01", "2024-12-31 23:00"),   # whatever 2024 CAMS has arrived
                           ("2025-01-01", "2025-08-31 23:00")]
    config.VAL_RANGE = ("2025-09-01", "2025-09-30 23:00")
    config.TEST_RANGE = ("2025-10-01", "2025-12-31 23:00")      # Oct-Nov 2025 stubble season
print("train:", config.TRAIN_RANGES, "| val:", config.VAL_RANGE, "| test:", config.TEST_RANGE,
      "| benchmark:", config.BENCHMARK_RANGE)

In [ ]:
# 4. Collect CAMS jobs that finished on the Copernicus servers overnight -> Drive
from data_pipeline import ingest_cams
status = ingest_cams.collect()
print(status)   # 'downloaded' = on Drive; 'running'/'accepted' = still on ADS (re-run this cell later)

In [ ]:
# 5. Preprocess (hourly alignment, gaps, CAMS -> stations) + features + graph  (~10-20 min on Drive)
from preprocessing import clean_align, features
from graph import build_graph
obs = clean_align.run()
feats = features.run()
graph = build_graph.run()

In [ ]:
# 6. Train. Epochs 0-4 teacher forcing, then scheduled sampling; horizon grows 24 -> 72 h.
from model import train
data = train.load_data()
print({s: len(data.split(s)) for s in ("train", "val", "test", "benchmark")})

overrides = {
    "epochs": 30, "batch_size": 8, "hidden": 64, "lr": 1e-3,
    "teacher_forcing_epochs": 5, "sampling_ramp_epochs": 15, "horizon_start": 24,
    "checkpoint_every": 5,
}
run_dir = train.run(overrides, data=data)

In [ ]:
# 7. Evaluate: per-lead RMSE/MAE vs raw CAMS, CPCB AQI categories, extreme recall,
#    coupling perturbation test, Jena et al. benchmark slice
from model import evaluate
metrics, metrics_path = evaluate.run(run_dir / "best.pt", data=data)

test = metrics["splits"]["test"]
if test.get("n_samples"):
    for d in ("day1", "day2", "day3"):
        e = test["errors"]["pm25"][d]
        print(f"PM2.5 {d}: VAAYU RMSE {e['model']['rmse']:.1f} vs CAMS {e['baseline']['rmse']:.1f} "
              f"({e['rmse_improvement_pct']:+.1f}%)")
    print("perturbation test:", test.get("perturbation_test"))
print("benchmark vs Jena 2021:", metrics["splits"].get("benchmark", {}).get("jena_comparison"))

In [ ]:
# 8. Promote to models/production/model.pt (only if better than the current one)
from model import promote
promote.run(run_dir / "best.pt", metrics_path)
print(open(config.REGISTRY_FILE).read())

### Full historical ingestion (not needed now — data is already on Drive)
`ingest_openaq`, `ingest_opencity`, `ingest_kaggle`, `ingest_openmeteo`, `ingest_era5`, `ingest_firms`
and `ingest_cams.submit(...)` are documented in the README.